# 📊 Step 3: Evaluation, Metrics & Error Analysis
### Project: Evaluating the Impact of RAG on Reducing LLM Hallucinations

In this notebook, we analyze:
1. **Aggregate Metric Comparisons** (Hallucination Rate, Faithfulness, Factual Accuracy)
2. **Category Breakdown** (Direct Fact vs Multi-Hop vs Out-of-Corpus vs Adversarial)
3. **Top-K & Prompt Strictness Ablations**
4. **Qualitative Case Studies & Failure Modes**

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append('..')

from src.config import RESULTS_FILE, PLOTS_DIR
from src.visualization import plot_hallucination_comparison, plot_category_breakdown, plot_ablation_comparison

df = pd.read_csv(RESULTS_FILE)
print(f"Loaded {len(df)} experiment rows.")
df.head(3)

## 1. Key Metric Calculations

In [ ]:
total = len(df)
summary = {
    "Baseline (No RAG)": {
        "Hallucination Rate": f"{(df['baseline_hallucinated'].sum()/total)*100:.1f}%",
        "Faithfulness": f"{df['baseline_faithfulness'].mean()*100:.1f}%"
    },
    "RAG (Top-3 Strict)": {
        "Hallucination Rate": f"{(df['rag_k3_hallucinated'].sum()/total)*100:.1f}%",
        "Faithfulness": f"{df['rag_k3_faithfulness'].mean()*100:.1f}%"
    },
    "RAG (Top-5 Strict)": {
        "Hallucination Rate": f"{(df['rag_k5_hallucinated'].sum()/total)*100:.1f}%
    },
    "RAG (Top-3 Loose)": {
        "Hallucination Rate": f"{(df['rag_loose_hallucinated'].sum()/total)*100:.1f}%
    }
}
pd.DataFrame(summary).T

## 2. Visualizing Benchmark Plots

In [ ]:
import matplotlib.image as mpimg

fig, ax = plt.subplots(figsize=(10, 6))
img = mpimg.imread(str(PLOTS_DIR / 'hallucination_reduction.png'))
ax.imshow(img)
ax.axis('off')
plt.show()

## 3. Qualitative Failure Case Analysis
Let's inspect questions where Baseline hallucinated vs how RAG resolved or handled it.

In [ ]:
halluc_cases = df[df['baseline_hallucinated'] == 1][['id', 'category', 'question', 'baseline_answer', 'rag_k3_answer', 'ground_truth']].head(5)
for _, r in halluc_cases.iterrows():
    print(f"\n[{r['id']}] Category: {r['category']}")
    print(f"Question: {r['question']}")
    print(f"🔴 Baseline: {r['baseline_answer']}")
    print(f"🟢 RAG: {r['rag_k3_answer']}")
    print(f"🎯 Ground Truth: {r['ground_truth']}")
    print("-"*60)